In [49]:
import numpy as np
import pandas as pd
import pickle
import ast
import networkx as nx

# ── config ────────────────────────────────────────────────────────────────────
ENOA_FILE   = "enoa/all_genres1.csv"        # path to ENAO data
OUT_MATRIX  = "enoa_graph.npy"                # output matrix
OUT_VOCAB   = "enoa_vocab.pkl"            # output genre list
# ──────────────────────────────────────────────────────────────────────────────

# load ENAO data
df = pd.read_csv(ENOA_FILE)
print(f"Loaded {len(df)} genres from ENAO")

Loaded 6291 genres from ENAO


In [50]:
# build edge list from SIM_GENRES and SIM_WEIGHTS
# each row is: genre -> similar_genre with a weight
# this is already directed (genre i -> j means j is similar to i)

edges = []  # list of (source, target, weight)

for _, row in df.iterrows():
    source = row['GENRE']
    try:
        sim_genres  = ast.literal_eval(row['SIM_GENRES'])
        sim_weights = ast.literal_eval(row['SIM_WEIGHTS'])
        for genre, weight in zip(sim_genres, sim_weights):
            edges.append((source, genre, float(weight)))
    except (ValueError, SyntaxError):
        continue

edges_df = pd.DataFrame(edges, columns=['source', 'target', 'weight'])

# remove self-loops
edges_df = edges_df[edges_df['source'] != edges_df['target']]

print(f"Total edges: {len(edges_df)}")
print(f"Sample edges:")
print(edges_df.head(10))

Total edges: 123439
Sample edges:
  source             target  weight
0    pop          dance pop   127.0
1    pop           boy band   108.0
2    pop  japanese teen pop   100.0
3    pop  finnish dance pop   100.0
4    pop             uk pop   107.0
5    pop          latin pop   100.0
6    pop        mexican pop   102.0
7    pop                edm   104.0
8    pop                r&b   102.0
9    pop          pop dance   105.0


In [51]:
# build genre index
genres = pd.unique(edges_df[['source', 'target']].values.ravel())
genre_to_idx = {g: i for i, g in enumerate(genres)}
n = len(genres)
print(f"Total unique genres: {n}")

# build adjacency matrix
adj = np.zeros((n, n), dtype=np.float32)

for _, row in edges_df.iterrows():
    i = genre_to_idx[row['source']]
    j = genre_to_idx[row['target']]
    adj[i, j] = max(adj[i, j], row['weight'])

print(f"Matrix shape: {adj.shape}")
print(f"Diagonal sum: {adj.diagonal().sum()}")   # should be 0
print(f"Is symmetric: {np.allclose(adj, adj.T)}") # should be False - directed!

Total unique genres: 6284
Matrix shape: (6284, 6284)
Diagonal sum: 0.0
Is symmetric: False


In [52]:
# find largest connected component
G = nx.from_numpy_array(adj)
components = list(nx.connected_components(G))
print(f"Number of connected components: {len(components)}")

largest_cc = max(components, key=len)
largest_cc = sorted(largest_cc)
print(f"Largest CC size: {len(largest_cc)}")

# extract LCC
adj_lcc    = adj[np.ix_(largest_cc, largest_cc)]
genres_lcc = [genres[i] for i in largest_cc]

print(f"LCC matrix shape: {adj_lcc.shape}")
print(f"LCC connected: {nx.is_connected(nx.from_numpy_array(adj_lcc))}")

Number of connected components: 1
Largest CC size: 6284
LCC matrix shape: (6284, 6284)
LCC connected: True


In [53]:
# normalize rows so weights are between 0 and 1
np.fill_diagonal(adj_lcc, 0)  # safety check
row_sums = adj_lcc.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
adj_lcc = adj_lcc / row_sums

print(f"After normalization:")
print(f"  Diagonal sum: {adj_lcc.diagonal().sum()}")    # should be 0
print(f"  Is symmetric: {np.allclose(adj_lcc, adj_lcc.T)}") # should be False
print(f"  Min value: {adj_lcc.min():.4f}")
print(f"  Max value: {adj_lcc.max():.4f}")
print(f"  Non-zero entries: {np.count_nonzero(adj_lcc)}")

After normalization:
  Diagonal sum: 0.0
  Is symmetric: False
  Min value: 0.0000
  Max value: 1.0000
  Non-zero entries: 123439


In [54]:
# check overlap with collaboration network
collab = pd.read_csv("code/dataset/Original/global/global-genre_network-2019.csv", sep='\t')
collab = collab[collab['source'] != collab['target']]
collab_genres = set(pd.unique(collab[['source', 'target']].values.ravel()))
enoa_genres   = set(genres_lcc)

overlap = collab_genres & enoa_genres
missing = collab_genres - enoa_genres

print(f"Collaboration network genres: {len(collab_genres)}")
print(f"ENOA LCC genres: {len(enoa_genres)}")
print(f"Overlap (usable for analysis): {len(overlap)}")
print(f"Missing from ENAO: {len(missing)}")
print(f"Missing genres: {sorted(missing)}")

Collaboration network genres: 310
ENOA LCC genres: 6284
Overlap (usable for analysis): 298
Missing from ENAO: 12
Missing genres: ['afro dancehall', 'baile pop', 'deep pop r&b', 'dutch urban', 'francoton', 'k-hop', 'latin', 'nc hip hop', 'ninja', 'regional mexican pop', 'trap espanol', 'vapor trap']


In [55]:
# remove nodes with no outgoing connections (empty rows)
row_sums_check = (adj_lcc > 0).sum(axis=1)
valid_nodes = np.where(row_sums_check > 0)[0]
isolated = np.where(row_sums_check == 0)[0]

print(f"Nodes with no outgoing connections: {len(isolated)}")
print(f"Isolated genres: {[genres_lcc[i] for i in isolated]}")

# filter them out
adj_lcc    = adj_lcc[np.ix_(valid_nodes, valid_nodes)]
genres_lcc = [genres_lcc[i] for i in valid_nodes]

print(f"Matrix shape after removing isolated nodes: {adj_lcc.shape}")

Nodes with no outgoing connections: 4
Isolated genres: ['pov: indie', 'lgbtq+ hip hop', 'pop lgbtq+ brasileira', 're:techno']
Matrix shape after removing isolated nodes: (6280, 6280)


In [56]:
# save matrix and vocab
np.save(OUT_MATRIX, adj_lcc)
pickle.dump(genres_lcc, open(OUT_VOCAB, 'wb'))

print(f"Saved matrix to {OUT_MATRIX}")
print(f"Saved vocab ({len(genres_lcc)} genres) to {OUT_VOCAB}")

Saved matrix to enoa_graph.npy
Saved vocab (6280 genres) to enoa_vocab.pkl
